# 02 — Chunk parsed documents

Turns the `layout.json` from Stage 1 into `data/chunks/<doc_id>.jsonl`.
Each line is a `Chunk` with its `sentences` array; `sentence_id`s are
doc-scoped and stable, and `text[char_start:char_end]` round-trips to
the sentence text on every record.

Chunking policy (set in `src/sentcite/chunking.py`):

- spaCy `sentencizer` (no model download).
- `tiktoken` `cl100k_base` for token counting (GPT-4o/4.1 family).
- Target ~400 tokens, ~40 token (≈10 %) overlap at sentence boundaries.
- Chunks never cross a section heading — keeps `section_path` faithful.

In [ ]:
import json, sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from sentcite.chunking import chunk_document

DOC_ID = 'irs_pub_587'
layout = json.loads((REPO_ROOT / 'data' / 'parsed' / DOC_ID / 'layout.json').read_text())
chunks = chunk_document(DOC_ID, layout)
{
    'doc': DOC_ID,
    'chunks': len(chunks),
    'sentences': sum(len(c.sentences) for c in chunks),
    'mean_tokens_per_chunk': round(sum(c.token_count for c in chunks) / max(len(chunks), 1), 1),
    'distinct_sections': len({tuple(c.section_path) for c in chunks}),
}

In [ ]:
# Verify sentence offsets round-trip inside each chunk.
bad = [
    (c.chunk_id, s.sentence_id)
    for c in chunks
    for s in c.sentences
    if c.text[s.char_start:s.char_end] != s.text
]
f'round-trip failures: {len(bad)}'

In [ ]:
# Peek at a mid-document chunk and its first few sentences.
c = chunks[len(chunks) // 3]
{
    'chunk_id': c.chunk_id,
    'page': c.page,
    'token_count': c.token_count,
    'section_path': c.section_path,
    'n_sentences': len(c.sentences),
    'first_3': [s.text for s in c.sentences[:3]],
}

In [ ]:
# Confirm sentence overlap across adjacent chunks within the same section:
# the last sentence_id of chunk N should reappear as the first sentence_id
# of chunk N+1 whenever they share a section_path.
examples = []
for a, b in zip(chunks, chunks[1:]):
    if tuple(a.section_path) != tuple(b.section_path):
        continue
    ids_a = {s.sentence_id for s in a.sentences[-5:]}
    ids_b = {s.sentence_id for s in b.sentences[:5]}
    overlap = ids_a & ids_b
    if overlap:
        examples.append((a.chunk_id, b.chunk_id, sorted(overlap)))
    if len(examples) >= 3:
        break
examples